# LeafDoc: Explainable Plant Disease Diagnosis & Agronomic Prescriptions

Welcome to the **LeafDoc** interactive demonstration notebook.

LeafDoc is a production-grade multi-stage diagnostic system that pairs deep learning with agronomic domain knowledge:
1. **Stage 1 (Binary Filter)**: Distinguishes Healthy vs Diseased tissue (`Macro-F1: 99.95%`).
2. **Stage 2 (Fine Classifier)**: Identifies 38 PlantVillage crop species and specific pathogens (`Accuracy: 99.9%`).
3. **Stage 3 (Explainability & Severity)**: Generates Grad-CAM activation maps and measures percentage of affected leaf surface area via dual-space HSV/LAB leaf segmentation.
4. **Stage 4 (Agronomic Prescriptions)**: Formulates tailored immediate actions, cultural controls, chemical fungicides, and biological remedies.

In [ ]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.inference.pipeline import LeafDocPipeline

print(f"Project root: {project_root}")

## 1. Initialize the LeafDoc Multi-Stage Pipeline

We instantiate `LeafDocPipeline`, which preloads Stage 1 and Stage 2 model checkpoints, initializes the PyTorch Grad-CAM hook engine, and loads the 38-class agronomic treatments encyclopedia.

In [ ]:
pipeline = LeafDocPipeline(
    config_path=str(project_root / "configs/config.yaml"),
    stage1_ckpt_path=str(project_root / "checkpoints/stage1_binary_best.pth"),
    stage2_ckpt_path=str(project_root / "checkpoints/stage2_fine_best.pth"),
    treatments_path=str(project_root / "data/treatments.json"),
)

print(f"Compute device: {pipeline.device}")
print(f"Total Fine-Grained Classes: {len(pipeline.s2_idx_to_class)}")
print(f"Total Treatment Records: {len(pipeline.recommender.treatments)}")

## 2. Diagnose a Diseased Leaf (Apple Scab)

In [ ]:
sample_diseased = project_root / "app/static/samples/apple_scab.jpg"

diagnosis_diseased = pipeline.diagnose(
    image_input=sample_diseased,
    generate_visualization=True,
    top_k=5
)

# Print formatted clinical report
pipeline.print_diagnosis_report(diagnosis_diseased)

### Visualizing the 4-Panel Diagnostic Dashboard

The 4-panel dashboard displays:
1. **Input Leaf Image**
2. **Grad-CAM Saliency Overlay** (shows neural attention on lesion biomarkers)
3. **Segmented Lesion Mask** (isolated disease contours on segmented leaf surface)
4. **Agronomic Diagnostic Card** (severity metric, affected area %, confidence rating)

In [ ]:
composite = diagnosis_diseased["visualizations"]["composite_panel"]

plt.figure(figsize=(12, 10), dpi=120)
plt.imshow(composite)
plt.axis("off")
plt.title("LeafDoc Comprehensive Diagnostic Dashboard", fontsize=14, pad=12)
plt.tight_layout()
plt.show()

## 3. Diagnose a Healthy Leaf (Peach Leaf)

In [ ]:
sample_healthy = project_root / "app/static/samples/peach_healthy.jpg"

diagnosis_healthy = pipeline.diagnose(
    image_input=sample_healthy,
    generate_visualization=True,
)

pipeline.print_diagnosis_report(diagnosis_healthy)

## 4. Detailed Inspection of Agronomic Treatment Prescription

LeafDoc separates remedies into actionable agricultural categories:

In [ ]:
treatment = diagnosis_diseased["stage4_treatment"]

print(f"=== Prescription: {treatment['condition_name']} ===")
print(f"Urgent Action    : {treatment['immediate_action']}\n")

print("Cultural Practices:")
for c in treatment['cultural_controls']:
    print(f"  • {c}")

print("\nChemical Sprays:")
for c in treatment['chemical_controls']:
    print(f"  • {c}")

print("\nBiological Controls:")
for b in treatment['biological_controls']:
    print(f"  • {b}")

print("\nLong-Term Prevention:")
for p in treatment['preventive_measures']:
    print(f"  • {p}")

## Conclusion

The LeafDoc pipeline is fully validated and ready for serving via FastAPI REST endpoints or the interactive web app (`app/main.py`).